# Vector 개념: `cutlass.Vector` 레지스터 vs 메모리

지금까지 데이터는 `cutlass.Array` 안에 있었습니다. 하지만 `Array`는 *메모리를 가리키는 핸들*일 뿐입니다 --
global, shared, local 공간을 가리키는 포인터입니다. 산술 자체는 **레지스터**에서 일어나고, 그 레지스터에
있는 것이 `cutlass.Vector`입니다: 스레드의 레지스터에 들어 있는 **불변 값**이고, 자기만의 주소는 없습니다.

load와 store가 둘을 잇습니다:

- **load** -- Array를 슬라이스하면 원소들이 Vector로 올라옵니다: `arr[base:V]`는 연속된 `V`개 원소를
  읽습니다 (레지스터 <- 메모리).
- **store** -- 슬라이스에 Vector를 대입하면 다시 씁니다: `arr[base:V] = vec` (메모리 <- 레지스터).

무엇을 쥐고 있느냐가 할 수 있는 일을 정합니다. `Array`는 인덱싱해서 제자리에서 대입할 수 있지만,
`Vector`는 *새* `Vector`로 변환하는 것만 됩니다.

원문: [`NVIDIA/cutlass` examples/python/CuTeDSL/notebooks/2_primitives/02_vector_concepts.ipynb](https://github.com/NVIDIA/cutlass/blob/098de2a652cf8f00fd70b2df54051c7eccbb855a/examples/python/CuTeDSL/notebooks/2_primitives/02_vector_concepts.ipynb) (BSD-3-Clause, commit `098de2a` 기준)

**배울 내용:** **Array(메모리) vs Vector(레지스터)** 구분; 슬라이스 `arr[base:V]`(`V`는 *개수*)가
`cutlass.Vector`를 로드하는 방식; **스칼라 브로드캐스트**가 있는 원소별 SIMD 산술(`vec * A + B`);
벡터화된 선택 `cutlass.vector.where`; `cutlass.vector.full`로 만드는 상수 벡터; 모든 레인을 스칼라 하나로
접는 레지스터 내 **수평 리덕션** `vec.reduce("add")` -- shared 메모리도 shuffle도 필요 없습니다.

**선수 지식:** 원문은 `01_array_concepts` 노트북(같은 `arr[base:V]` 슬라이스)을 선수로 적었습니다. 이 저장소에서는
`05_array_concepts.ipynb`가 그 노트북입니다.

**실행 환경:** 이 노트북은 커널을 *실제로 launch합니다*. 노트북4·5와 마찬가지로 **`CUTE_DSL_ARCH` 오버라이드를
넣지 않습니다**. `Arch` enum이 `sm_80`부터 시작하므로 T4는 해당 없습니다(README 참고). 이 노트북은
**Colab L4**에서 실행해 검증했습니다.

## 0. 환경 확인

재현성을 위해 먼저 GPU 정보를 남깁니다.

In [ ]:
!nvidia-smi

## 0-1. CuTe DSL 설치

`nvidia-cutlass-dsl==4.8.0`으로 고정합니다(노트북1과 동일 버전).

**함정(노트북1에서 이미 확인함): 같은 커널에서 설치 직후 import하면 실패합니다.** `.pth`
파일이 인터프리터 시작 시점에만 읽히기 때문 — 설치 셀 다음에 반드시 `site.main()` 셀을
넣습니다.

In [ ]:
!pip install -q nvidia-cutlass-dsl==4.8.0

In [ ]:
import site
site.main()  # 방금 설치된 nvidia-cutlass-dsl의 .pth 경로를 sys.path에 반영

In [ ]:
import cutlass
import cutlass.cute as cute
import torch

### 함정: `cute.printf` 출력이 셀에 기본적으로 안 뜹니다 (노트북1·2 참고)

노트북1·2에서 확인한 대로, `cute.printf`(디바이스 출력) 결과가 이 Colab 환경에서는 셀 출력에
기본적으로 나타나지 않습니다(원인은 확인 못 함 -- 노트북1의 실측 검증 절 참고). 이 챕터의 커널은 Vector
연산 단계마다 `cute.printf`로 첫 타일의 값을 찍어 변환 과정을 보여주는 게 핵심이라, 그대로 두면 그 줄들이
전부 빠집니다. 노트북1·2의 `show_device_printf()` 헬퍼를 그대로 가져와 씁니다.

In [ ]:
import os
import sys
import ctypes
import tempfile
import contextlib


@contextlib.contextmanager
def show_device_printf():
    """cute.printf가 fd 1에 직접 쓰는 내용을 캡처해서 with 블록이 끝난 뒤 print합니다."""
    sys.stdout.flush()
    saved_fd1 = os.dup(1)
    tmp = tempfile.NamedTemporaryFile(delete=False)
    tmp_path = tmp.name
    tmp.close()
    tmp_fd = os.open(tmp_path, os.O_WRONLY)
    os.dup2(tmp_fd, 1)
    try:
        yield
        cutlass.cuda.stream_sync(cutlass.cuda.default_stream())
        ctypes.CDLL(None).fflush(None)
    finally:
        sys.stdout.flush()
        os.dup2(saved_fd1, 1)
        os.close(saved_fd1)
        os.close(tmp_fd)
    with open(tmp_path, "r") as f:
        captured = f.read()
    os.unlink(tmp_path)
    if captured:
        print(captured, end="")

## 1. Array는 메모리, Vector는 레지스터

| 타입 | 무엇인가 | 연산 |
|---|---|---|
| `cutlass.Array` | 메모리를 가리키는 핸들 (global / shared / local) | `arr[i]` 스칼라 load/store; `arr[i:V]` 벡터 load -> `Vector`; `arr[i:V] = vec` 벡터 store (레지스터 -> 메모리) |
| `cutlass.Vector` | 스레드의 레지스터에 든 불변 값 | `vec * A + B` 원소별 + 브로드캐스트; `cutlass.vector.where(c, a, b)` 벡터화 선택; `vec.reduce("add")` 수평 접기 -> 스칼라 |

`Vector`는 **저장소가 아니라 값**입니다: 원소 대입이 없습니다(`vec[i] = x`는 `TypeError`). `Vector`를
"바꾸려면" *새* 것을 만드는 연산을 적용합니다 -- 파이썬 `int`에 더하는 것과 똑같습니다.

아래 커널이 전체 왕복을 돕니다. 스레드마다 `V`-폭 타일 하나를 Vector로 로드하고, `act(A*x + B)`를 전부
레지스터 안에서 계산하고 -- 여기서 `act`는 트레이스 시점에 `OPS` **딕셔너리**에서 고릅니다 -- 결과를
저장하고, `reduce`로 타일을 스칼라 하나로 접기도 합니다.

In [ ]:
@cute.kernel
def vector_concepts_kernel(
    x: cutlass.Array, y: cutlass.Array, s: cutlass.Array,
    N: cutlass.Int32, A: cutlass.Constexpr, B: cutlass.Constexpr, V: cutlass.Constexpr,
    act: cutlass.Constexpr = "relu",
):
    tx, _, _ = cute.arch.thread_idx()
    bx, _, _ = cute.arch.block_idx()
    bdx, _, _ = cute.arch.block_dim()
    tidx = bx * bdx + tx
    base = tidx * V                      # this thread owns the V-wide tile starting here

    if base < N:
        # Step 1. Load: one V-wide vector load pulls the tile into registers.
        # A cute.printf after each Vector op makes the per-stage transform visible (tile 0 only).
        vec = x[base:V]                  # memory -> registers
        if tidx == 0:
            cute.printf(f"load    x[base:V]  = [{vec[0]} {vec[1]} {vec[2]} {vec[3]}]")

        # Step 2. Affine map: scalars A, B broadcast across all V lanes (SIMD).
        affine = vec * A + B
        if tidx == 0:
            cute.printf(f"affine  vec*A + B  = [{affine[0]} {affine[1]} {affine[2]} {affine[3]}]")

        # Step 3. Activation. OPS maps a name to a vector op; `act` is a Constexpr, so
        # OPS[act] picks the op at trace time and the kernel bakes in only that one.
        zeros = cutlass.vector.full((V,), 0.0, dtype=cutlass.Float32)
        OPS = {
            "relu":   lambda u: cutlass.vector.where(u > zeros, u, zeros),
            "square": lambda u: u * u,
            "abs":    lambda u: cutlass.vector.where(u > zeros, u, zeros - u),
        }
        activated = OPS[act](affine)
        if tidx == 0:
            cute.printf(f"act ({act}) = [{activated[0]} {activated[1]} {activated[2]} {activated[3]}]")

        # Step 4. Horizontal fold: V lanes -> one scalar, in registers.
        total = activated.reduce("add")
        if tidx == 0:
            cute.printf(f"reduce  sum        = {total}")

        # Step 5. Store: one V-wide vector store writes the tile back, plus the per-tile sum.
        y[base:V] = activated            # registers -> memory
        s[tidx] = total

## 2. Launch: 스레드당 V-폭 타일 하나

그리드는 128스레드 블록으로 `N // V`개 타일을 전부 덮습니다. `A`, `B`, `V`는 `Constexpr`이라서 계수가 상수로
접히고, load / store / reduce가 고정 폭 명령어가 됩니다.

In [ ]:
@cute.jit
def vector_concepts(
    x: cutlass.Array, y: cutlass.Array, s: cutlass.Array,
    N: cutlass.Int32, A: cutlass.Constexpr, B: cutlass.Constexpr, V: cutlass.Constexpr,
    act: cutlass.Constexpr = "relu",
):
    block = (128, 1, 1)
    tiles = N // V                                       # one thread per V-wide tile
    grid = ((tiles + block[0] - 1) // block[0], 1, 1)
    vector_concepts_kernel(x, y, s, N, A, B, V, act).launch(grid=grid, block=block)

## 3. 실행하고 PyTorch와 비교하기

`cutlass.Array` 매개변수는 `cute.runtime.from_dlpack`으로 PyTorch CUDA 텐서를 바로 받습니다(복사 없음).
`OPS` 딕셔너리의 활성화 함수마다 같은 커널을 돌리고, 두 출력 -- 원소별 타일, 그리고 타일별 행 합 -- 을 대응하는
PyTorch 연산과 비교합니다.

In [ ]:
N, V = 1 << 20, 4          # 1,048,576 elements, one 4-wide tile per thread
A, B = 2.0, 1.0            # y = act(A*x + B)

x = torch.randn(N, dtype=torch.float32, device="cuda")

# The SAME kernel, specialized to each activation in OPS by name. `act` is a Constexpr,
# so each launch traces just one op out of the dict -- a compile-time dispatch table.
refs = {"relu": torch.relu, "square": lambda t: t * t, "abs": torch.abs}
with show_device_printf():
    for act in ("relu", "square", "abs"):
        y = torch.zeros(N, dtype=torch.float32, device="cuda")
        s = torch.zeros(N // V, dtype=torch.float32, device="cuda")   # one row-sum per tile
        vector_concepts(
            cute.runtime.from_dlpack(x), cute.runtime.from_dlpack(y), cute.runtime.from_dlpack(s),
            N, A, B, V, act,
        )
        ref = refs[act](A * x + B)
        torch.testing.assert_close(y.cpu(), ref.cpu(), atol=1e-2, rtol=1e-2)
        torch.testing.assert_close(s.cpu(), ref.reshape(N // V, V).sum(dim=-1).cpu(), atol=1e-2, rtol=1e-2)
        print(f"PASS  act={act}")

# Expected output (tile 0 prints once per activation; lane values depend on the random input):
# load    x[base:V]  = [...]
# affine  vec*A + B  = [...]
# act (relu) = [...]
# reduce  sum        = ...
# PASS  act=relu
# ... then act=square, act=abs ...

### 실측 검증 (2026-10-01, Colab L4)

이 노트북 전체를 `.ipynb` 단위로(실제 셀 순서대로 하나의 커널에서) Colab **L4**에서 실행해
확인했습니다. 코드 셀 8개가 전부 에러 없이 통과했습니다(`nvidia-cutlass-dsl==4.8.0`). 노트북4·5처럼
`CUTE_DSL_ARCH` 오버라이드 없이 기본 감지(sm_89)로 돌렸습니다.

| 확인한 것 | 출력 | 원문 기대 출력과 |
|---|---|---|
| 세 활성화 함수 모두 PyTorch와 일치 | `PASS  act=relu`, `PASS  act=square`, `PASS  act=abs` | 일치 |
| 첫 타일(활성화마다 한 번) | `load`, `affine`, `act (...)`, `reduce  sum` 네 줄이 각 활성화마다 출력 | 일치 (원문은 값을 `[...]`로 생략) |
| 리덕션 검산 | relu: `1.382353 + 0.000000 + 1.521302 + 2.607111 = 5.510766` = 출력된 `reduce sum`, square: `16.629433`, abs: `7.878707` 역시 네 원소의 합과 일치 | 일치 |

출력 예(입력이 `torch.randn`이라 실행마다 값이 다릅니다): `load    x[base:V]  = [0.191176 -1.683971 0.260651 0.803556]`
→ `affine  vec*A + B  = [1.382353 -2.367941 1.521302 2.607111]` (= `2.0·x + 1.0`) → `act (relu) = [1.382353 0.000000 1.521302 2.607111]`.
Vector 원소를 넣은 `cute.printf`의 f-string은 `[a b c d]` 형태(소수점 6자리)로 출력됐습니다.

**실행 셀 수정:** 섹션 3의 실행 셀에서 `for` 루프 전체를 `with show_device_printf():`로 감쌌습니다(들여쓰기만 바뀌었고
나머지는 원문 그대로). 나머지 코드 셀 3개는 원문 그대로입니다. 이 노트북에서 헬퍼 *없이* 돌려서 디바이스 줄이 실제로
빠지는지는 따로 확인하지 않았습니다 -- 노트북1·2에서 확인한 현상을 근거로 미리 적용했습니다.

**출력 순서(관찰):** 호스트 `PASS  act=...` 세 줄이 먼저 나오고, 그 *뒤에* 디바이스 출력 12줄(활성화 3개 × 4줄)이 활성화 순서대로
묶여서 나왔습니다. 원문 주석은 활성화마다 네 줄 다음에 `PASS`가 오는 식으로 적었지만, 헬퍼가 `with` 블록이 끝날 때 캡처한
디바이스 출력을 한꺼번에 `print`하기 때문에 이 순서가 되는 것은 셀 코드에서 그대로 따라 나옵니다. 내용은 같고 순서만 다릅니다.

**검증의 한계:** L4(sm_89)에서 한 번 실행한 결과입니다. "직접 해보기"의 변형(fp16 변환, `reduce("max")` 등)은 실행하지 않았습니다.


## 직접 해보기

1. **Vector는 불변입니다.** `vec[0] = cutlass.Float32(9.0)`을 추가해 보세요 -- `TypeError`가 납니다,
   Vector는 저장소가 아니라 값이니까요. 그걸로 *새* Vector를 만드는 것만 됩니다.
2. **활성화 함수 추가하기.** `OPS` 딕셔너리에 `"gelu": lambda u: ...`와 대응하는 참조 구현을 넣어 보세요 --
   딕셔너리는 트레이스 시점 디스패치 표라서, 항목 하나를 더하면 특수화된 커널이 공짜로 하나 생깁니다.
3. **다른 리덕션.** `reduce("add")`를 `reduce("max")`로 바꿔 보세요(torch 참조도 맞춰서);
   `reduce("mul")`과 `reduce("min")`도 됩니다.
4. **스칼라로 Vector 만들기.** load를
   `vec = cutlass.Vector.from_elements((1.0, 2.0, 3.0, 4.0), cutlass.Float32)`로 바꿔 보세요 -- Vector가
   꼭 메모리에서 와야 하는 건 아닙니다.
5. **타입 변환.** affine 연산 앞에 `vec = vec.to(cutlass.Float16)`을 넣어 보세요(`fp16` 출력에 저장하고
   허용 오차를 넓혀서). `Vector.to(...)`는 원소 타입을 변환하고, `.bitcast(...)`는 같은 비트를 다른 타입으로
   재해석합니다.